# AI Learning Assistant - RAG Prototype (LangChain Runnable)

This notebook follows the project instructions and attached system design exactly:

1. Ingest PDFs and notebooks from `data/raw`
2. Chunk text with LangChain `RecursiveCharacterTextSplitter` (500 / 100)
3. Build three Chroma indexes:
   - Courses
   - Parts (with `course` metadata)
   - Chunks (with `course`, `section`, `source_file`, `chunk_id`)
4. Query pipeline:
   - Retrieve course
   - Retrieve parts filtered by course
   - Retrieve top 20 chunks filtered by course
   - Rerank with `BAAI/bge-reranker-base` and keep top 5
   - Compress context and generate answer with `llama3.1`
5. Guardrails are enforced in the final prompt:
   - Out-of-scope questions are handled politely
   - Broad/ambiguous questions request clarification
   - Answers stay grounded in retrieved context

In [13]:
from __future__ import annotations

import ast
import json
import os
import re
import signal
import subprocess
import sys
import tempfile
from pathlib import Path
from typing import Any, TypedDict, cast

import langchain
from chromadb import PersistentClient
from chromadb.api.shared_system_client import SharedSystemClient
from chromadb.utils import embedding_functions
from langchain_core.documents import Document
from langchain_core.globals import set_debug, set_verbose
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph
from nbformat import read as nb_read
from pypdf import PdfReader
from sentence_transformers import CrossEncoder

langchain.debug = True
set_debug(True)
set_verbose(True)

EMBEDDING_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.1"
RERANKER_MODEL = "BAAI/bge-reranker-base"

CHUNK_SIZE_TOKENS = 500
CHUNK_OVERLAP_TOKENS = 100
TOP_K_COURSES = 3
TOP_K_PARTS = 8
TOP_K_CHUNKS = 20
TOP_K_FINAL = 5

In [2]:
def clean_course_name(file_path: Path) -> str:
    """Map file name to a normalized course name."""
    name = file_path.stem.lower().strip()
    name = re.sub(r"[_\-]+", " ", name)
    name = re.sub(r"\s+", " ", name)
    return name


def extract_pdf_text(file_path: Path) -> str:
    """Extract text from PDF and keep pages connected for cross-page chunks."""
    reader = PdfReader(str(file_path))
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(pages)


def extract_notebook_text(file_path: Path) -> str:
    """Extract all notebook cell content as plain text."""
    with file_path.open("r", encoding="utf-8") as file:
        nb = nb_read(file, as_version=4)

    blocks: list[str] = []
    for cell in nb.cells:
        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)
        if source.strip():
            blocks.append(source)
    return "\n\n".join(blocks)


def normalize_chunk_text(text: str) -> str:
    """Normalize chunk text for retrieval while preserving punctuation."""
    normalized = text.replace("\r\n", "\n").replace("\r", "\n")
    normalized = re.sub(r"(?<=\w)-\s*\n\s*(?=\w)", "", normalized)
    normalized = re.sub(r"\s*\n+\s*", " ", normalized)
    normalized = re.sub(r"\s+", " ", normalized).strip()
    return normalized.lower()


def detect_sections(text: str) -> list[tuple[str, str]]:
    """Split by numbered headings like '1.intro' or fallback to one section."""
    pattern = re.compile(r"^\s*(\d+\.[^\n]+)", re.MULTILINE)
    matches = list(pattern.finditer(text))

    if not matches:
        return [("0.general", text)]

    sections: list[tuple[str, str]] = []
    for i, match in enumerate(matches):
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        section_name = match.group(1).strip().lower()
        section_text = text[start:end].strip()
        sections.append((section_name, section_text))
    return sections


def build_chunker() -> RecursiveCharacterTextSplitter:
    """LangChain recursive chunker with 500/100 token-like sizing."""
    return RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE_TOKENS,
        chunk_overlap=CHUNK_OVERLAP_TOKENS,
        length_function=lambda txt: len(txt.split()),
        separators=["\n\n", "\n", ". ", " ", ""],
    )


def build_documents(raw_data_dir: Path) -> tuple[list[Document], list[Document], list[Document]]:
    """Create documents for course, part, and chunk collections."""
    chunker = build_chunker()

    course_docs: list[Document] = []
    part_docs: list[Document] = []
    chunk_docs: list[Document] = []

    seen_courses: set[str] = set()
    seen_parts: set[tuple[str, str]] = set()

    for file_path in raw_data_dir.rglob("*"):
        if not file_path.is_file() or file_path.suffix.lower() not in {".pdf", ".ipynb"}:
            continue

        course = clean_course_name(file_path)
        if course not in seen_courses:
            seen_courses.add(course)
            course_docs.append(Document(page_content=course, metadata={"course": course}))

        full_text = extract_pdf_text(file_path) if file_path.suffix.lower() == ".pdf" else extract_notebook_text(file_path)
        for section, section_text in detect_sections(full_text):
            part_key = (course, section)
            if part_key not in seen_parts:
                seen_parts.add(part_key)
                part_docs.append(
                    Document(page_content=section, metadata={"course": course, "section": section})
                )

            for idx, chunk_text in enumerate(chunker.split_text(section_text)):
                normalized_chunk = normalize_chunk_text(chunk_text)
                if not normalized_chunk:
                    continue

                chunk_docs.append(
                    Document(
                        page_content=normalized_chunk,
                        metadata={
                            "course": course,
                            "section": section,
                            "source_file": file_path.name,
                            "chunk_id": f"chunk::{course}::{section}::{idx}",
                        },
                    )
                )

    return course_docs, part_docs, chunk_docs

In [3]:
def locate_workspace_root() -> Path:
    """Resolve workspace root when running from backend/rag notebook folder."""
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "raw").exists():
        return cwd
    if cwd.name == "rag" and (cwd.parent.parent / "data" / "raw").exists():
        return cwd.parent.parent
    return cwd


def create_chroma_client(persist_dir: Path) -> PersistentClient:
    """Create Chroma client for a persistent local vector DB."""
    return PersistentClient(path=str(persist_dir))


def reset_collections(persist_dir: Path, collection_names: list[str]) -> None:
    """Drop prototype collections to avoid duplicate indexing."""
    SharedSystemClient.clear_system_cache()
    client = create_chroma_client(persist_dir)
    for name in collection_names:
        try:
            client.delete_collection(name)
        except ValueError:
            pass


def upsert_documents(
    collection: Any,
    documents: list[Document],
    id_prefix: str,
    batch_size: int = 64,
 ) -> None:
    """Insert notebook-built documents into a Chroma collection in batches."""
    if not documents:
        return

    for start in range(0, len(documents), batch_size):
        batch = documents[start : start + batch_size]
        ids: list[str] = []
        texts: list[str] = []
        metadatas: list[dict[str, Any]] = []

        for offset, doc in enumerate(batch):
            ids.append(f"{id_prefix}::{start + offset}")
            texts.append(doc.page_content)
            metadatas.append(doc.metadata)

        collection.add(ids=ids, documents=texts, metadatas=metadatas)


workspace_root = locate_workspace_root()
raw_data_dir = workspace_root / "data" / "raw"
persist_dir = workspace_root / "backend" / "vectordb" / "chroma_db"
persist_dir.mkdir(parents=True, exist_ok=True)

reset_collections(
    persist_dir,
    ["prototype_courses", "prototype_parts", "prototype_chunks"],
)

embedding_fn = embedding_functions.OllamaEmbeddingFunction(
    model_name=EMBEDDING_MODEL,
    url="http://localhost:11434/api/embeddings",
)
client = create_chroma_client(persist_dir)

course_collection = client.get_or_create_collection(
    name="prototype_courses",
    embedding_function=embedding_fn,
)
part_collection = client.get_or_create_collection(
    name="prototype_parts",
    embedding_function=embedding_fn,
)
chunk_collection = client.get_or_create_collection(
    name="prototype_chunks",
    embedding_function=embedding_fn,
)

course_docs, part_docs, chunk_docs = build_documents(raw_data_dir)
upsert_documents(course_collection, course_docs, "course")
upsert_documents(part_collection, part_docs, "part")
upsert_documents(chunk_collection, chunk_docs, "chunk")

reranker = CrossEncoder(RERANKER_MODEL)

print(f"Indexed courses: {len(course_docs)}")
print(f"Indexed parts:   {len(part_docs)}")
print(f"Indexed chunks:  {len(chunk_docs)}")

Indexed courses: 32
Indexed parts:   690
Indexed chunks:  712


In [4]:
CONTEXT_TOKEN_LIMIT = 5000
REVIEW_TIMEOUT_SECONDS = 5
PYLINT_TIMEOUT_SECONDS = 20
COMPLEXITY_THRESHOLD = 10
MAX_LINT_MESSAGES = 25

chat_llm = ChatOllama(model=LLM_MODEL, temperature=0.0, verbose=True)

try:
    import resource
except ImportError:
    resource = None


def _parse_guard_response(raw_output: str) -> dict[str, str]:
    """Parse guard LLM output and normalize status/message."""
    default_message = "Your question is broad or ambiguous. Please narrow it to a specific topic, section, or example."
    cleaned = raw_output.strip()
    cleaned = re.sub(r"^```(?:json)?\\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\\s*```$", "", cleaned)

    try:
        payload = json.loads(cleaned)
        status = str(payload.get("status", "")).strip().lower()
        if status == "ok":
            return {"status": "ok"}
        if status in {"needs_clarification", "clarify", "ambiguous", "stop"}:
            message = str(payload.get("message", "")).strip()
            return {"status": "stop", "message": message or default_message}
    except json.JSONDecodeError:
        pass

    lowered = cleaned.lower()
    if "needs_clarification" in lowered or "clarify" in lowered or "ambiguous" in lowered:
        return {"status": "stop", "message": default_message}
    if '"status"' in lowered and '"ok"' in lowered:
        return {"status": "ok"}

    # Default open to avoid false rejects when model formatting drifts.
    return {"status": "ok"}


def guard_question(question: str) -> dict[str, Any]:
    """Use the model to decide whether clarification is required."""
    guard_prompt = f"""
You classify whether a user question is clear enough for retrieval in a data-science learning assistant.

Return JSON only with one of these exact formats:
{{"status":"ok"}}
{{"status":"needs_clarification","message":"<one short clarifying request>"}}

Decision rule:
- Choose "ok" if the question has a concrete topic and can reasonably be answered, even when short.
- Choose "needs_clarification" only when user intent is genuinely unclear or too broad to answer usefully.

Question:
{question}
""".strip()

    response = chat_llm.invoke(guard_prompt)
    decision = _parse_guard_response(str(response.content))
    if decision.get("status") == "ok":
        return {"status": "ok", "question": question}

    return {
        "status": "stop",
        "message": decision.get(
            "message",
            "Your question is broad or ambiguous. Please narrow it to a specific topic, section, or example.",
        ),
    }


def normalize_retrieval_query(question: str) -> str:
    """Normalize retrieval query without stripping punctuation."""
    normalized = question.lower().replace("\r\n", "\n").replace("\r", "\n")
    normalized = re.sub(r"\s*\n+\s*", " ", normalized)
    normalized = re.sub(r"\s+", " ", normalized).strip()
    return normalized or question.strip()


def query_collection(
    collection: Any,
    question: str,
    n_results: int,
    where: dict[str, Any] | None = None,
) -> list[Document]:
    """Query a Chroma collection and map results to LangChain Documents."""
    query_kwargs: dict[str, Any] = {
        "query_texts": [question],
        "n_results": n_results,
    }
    if where is not None:
        query_kwargs["where"] = where

    result = collection.query(**query_kwargs)
    documents = result.get("documents", [[]])[0]
    metadatas = result.get("metadatas", [[]])[0]
    return [
        Document(page_content=doc_text, metadata=metadata or {})
        for doc_text, metadata in zip(documents, metadatas)
    ]


def rerank_documents(question: str, documents: list[Document], top_k: int = TOP_K_FINAL) -> list[Document]:
    """Rerank with bge-reranker-base and keep top-k."""
    if not documents:
        return []

    pairs = [[question, doc.page_content] for doc in documents]
    scores = reranker.predict(pairs)
    ordered_indices = sorted(
        range(len(documents)),
        key=lambda idx: float(scores[idx]),
        reverse=True,
    )
    return [documents[idx] for idx in ordered_indices[:top_k]]


def approximate_tokens(text: str) -> int:
    """Token approximation using whitespace-separated units."""
    return len(text.split())


def format_context_block(doc: Document, index: int, body: str) -> str:
    """Format one context block with source metadata."""
    meta = doc.metadata
    return (
        f"[{index}] course={meta['course']} | section={meta['section']} | "
        f"source={meta['source_file']} | chunk_id={meta['chunk_id']}\\n{body}"
    )


def summarize_chunk_with_llm(question: str, doc: Document, index: int) -> str:
    """Summarize contextual ideas only while preserving formulas and algorithms."""
    prompt = f"""
You are compressing a retrieved context chunk for a RAG system.

Rules:
1. Preserve every formula, equation, and mathematical expression exactly as written.
2. Preserve every algorithm step, pseudocode, and code instruction exactly as written.
3. Summarize only explanatory/contextual ideas around formulas and algorithms.
4. Do not invent facts and do not add external knowledge.
5. Keep output concise and faithful to the chunk.

Question:
{question}

Chunk metadata:
course={doc.metadata['course']}, section={doc.metadata['section']}, source={doc.metadata['source_file']}, chunk_id={doc.metadata['chunk_id']}

Chunk text:
{doc.page_content}
""".strip()

    response = chat_llm.invoke(prompt)
    summary = response.content.strip()
    return format_context_block(doc, index, summary)


def compress_context(question: str, documents: list[Document], token_limit: int = CONTEXT_TOKEN_LIMIT) -> str:
    """Summarize chunks with LLM when context exceeds token limit."""
    full_blocks = [
        format_context_block(doc, i, doc.page_content)
        for i, doc in enumerate(documents, start=1)
    ]
    full_context = "\\n\\n".join(full_blocks)

    if approximate_tokens(full_context) <= token_limit:
        return full_context

    summarized_blocks = [
        summarize_chunk_with_llm(question, doc, i)
        for i, doc in enumerate(documents, start=1)
    ]
    return "\\n\\n".join(summarized_blocks)


def retrieve_payload(payload: dict[str, Any]) -> dict[str, Any]:
    """Course -> parts -> chunks retrieval, then rerank and compress."""
    question = payload["question"]
    retrieval_question = normalize_retrieval_query(question)

    # Anchor routing on chunk-level semantic hits to avoid course-name misrouting.
    global_chunk_hits = query_collection(
        chunk_collection,
        retrieval_question,
        TOP_K_CHUNKS,
    )
    if not global_chunk_hits:
        return {"status": "stop", "message": "I do not have enough retrieved context to answer this reliably."}

    top_global_course = str(global_chunk_hits[0].metadata.get("course", "")).strip()
    course_hits = query_collection(course_collection, retrieval_question, TOP_K_COURSES)
    selected_course = top_global_course or (course_hits[0].page_content if course_hits else "")
    if not selected_course:
        return {"status": "stop", "message": "No indexed course matched this question."}

    part_hits = query_collection(
        part_collection,
        retrieval_question,
        TOP_K_PARTS,
        where={"course": selected_course},
    )
    selected_parts = [doc.page_content for doc in part_hits]

    chunk_hits = [doc for doc in global_chunk_hits if doc.metadata.get("course") == selected_course]
    if not chunk_hits:
        chunk_hits = query_collection(
            chunk_collection,
            retrieval_question,
            TOP_K_CHUNKS,
            where={"course": selected_course},
        )

    if selected_parts:
        part_set = set(selected_parts)
        part_filtered = [doc for doc in chunk_hits if doc.metadata.get("section") in part_set]
        if part_filtered:
            chunk_hits = part_filtered

    ranked_docs = rerank_documents(question, chunk_hits, top_k=TOP_K_FINAL)
    if not ranked_docs:
        return {"status": "stop", "message": "I do not have enough retrieved context to answer this reliably."}

    context = compress_context(question, ranked_docs)
    return {
        "status": "ok",
        "question": question,
        "selected_course": selected_course,
        "selected_parts": selected_parts,
        "ranked_docs": ranked_docs,
        "context": context,
    }


def extract_task_and_code(query: str) -> tuple[str, str]:
    """Extract fenced Python code from a user query."""
    code_pattern = re.compile(r"```(?:python)?\\s*([\\s\\S]*?)```", re.IGNORECASE)
    match = code_pattern.search(query)
    if not match:
        return query.strip(), ""

    user_code = match.group(1).strip()
    user_task = code_pattern.sub("", query).strip()
    if not user_task:
        user_task = "Review this Python code."
    return user_task, user_code


def choose_tool_route(query: str, has_code: bool) -> str:
    """Route to code reviewer when code/debug intent is detected."""
    if has_code:
        return "code_reviewer"

    lowered = query.lower()
    code_hints = [
        "traceback",
        "stack trace",
        "stderr",
        "stdout",
        "review code",
        "fix my code",
        "debug",
        "exception",
        "python code",
        "pylint",
        "why does this code fail",
    ]
    if any(hint in lowered for hint in code_hints):
        return "code_reviewer"
    return "rag_agent"


def _to_text(value: str | bytes | None) -> str:
    if value is None:
        return ""
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="ignore")
    return value


def extract_traceback(stderr: str) -> str:
    """Extract traceback block from stderr when present."""
    marker = "Traceback (most recent call last):"
    if marker not in stderr:
        return ""
    return stderr[stderr.find(marker) :].strip()


def _sandbox_preexec_fn() -> None:
    """Apply lightweight process limits on POSIX systems."""
    if resource is None:
        return

    memory_limit_bytes = 512 * 1024 * 1024
    cpu_limit_seconds = REVIEW_TIMEOUT_SECONDS
    resource.setrlimit(resource.RLIMIT_AS, (memory_limit_bytes, memory_limit_bytes))
    resource.setrlimit(resource.RLIMIT_CPU, (cpu_limit_seconds, cpu_limit_seconds))
    signal.signal(signal.SIGXCPU, signal.SIG_DFL)


def run_pylint_report(user_code: str) -> dict[str, Any]:
    """Run pylint via subprocess and return parsed JSON messages."""
    temp_path: str | None = None
    try:
        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False, encoding="utf-8") as temp_file:
            temp_file.write(user_code)
            temp_path = temp_file.name

        process = subprocess.run(
            [
                sys.executable,
                "-m",
                "pylint",
                temp_path,
                "--output-format=json",
                "--score=n",
                "--reports=n",
            ],
            capture_output=True,
            text=True,
            timeout=PYLINT_TIMEOUT_SECONDS,
        )

        stdout = (process.stdout or "").strip()
        stderr = (process.stderr or "").strip()

        messages: list[dict[str, Any]] = []
        if stdout:
            try:
                payload = json.loads(stdout)
                if isinstance(payload, list):
                    for item in payload[:MAX_LINT_MESSAGES]:
                        messages.append(
                            {
                                "type": item.get("type", ""),
                                "symbol": item.get("symbol", ""),
                                "line": item.get("line", None),
                                "message": item.get("message", ""),
                            }
                        )
            except json.JSONDecodeError:
                return {
                    "messages": [],
                    "error": f"Unable to parse pylint JSON output: {stdout[:400]}",
                }

        return {
            "messages": messages,
            "error": stderr[:800] if stderr else "",
        }
    except subprocess.TimeoutExpired:
        return {
            "messages": [],
            "error": f"pylint timed out after {PYLINT_TIMEOUT_SECONDS} seconds.",
        }
    except Exception as exc:
        return {
            "messages": [],
            "error": f"pylint execution failed: {exc}",
        }
    finally:
        if temp_path:
            Path(temp_path).unlink(missing_ok=True)


class CodeStructureAnalyzer(ast.NodeVisitor):
    """AST-based checks for unreachable code and deep loop nesting."""

    def __init__(self) -> None:
        self.unreachable_code: list[dict[str, Any]] = []
        self.nested_loops: list[dict[str, Any]] = []
        self._loop_depth = 0

    def _scan_block(self, body: list[ast.stmt]) -> None:
        terminated = False
        for stmt in body:
            if terminated:
                self.unreachable_code.append(
                    {
                        "line": getattr(stmt, "lineno", None),
                        "node": type(stmt).__name__,
                    }
                )
            self.visit(stmt)
            if isinstance(stmt, (ast.Return, ast.Raise, ast.Break, ast.Continue)):
                terminated = True

    def visit_Module(self, node: ast.Module) -> None:
        self._scan_block(node.body)

    def visit_FunctionDef(self, node: ast.FunctionDef) -> None:
        self._scan_block(node.body)

    def visit_AsyncFunctionDef(self, node: ast.AsyncFunctionDef) -> None:
        self._scan_block(node.body)

    def visit_For(self, node: ast.For) -> None:
        self._loop_depth += 1
        if self._loop_depth >= 2:
            self.nested_loops.append(
                {
                    "line": node.lineno,
                    "depth": self._loop_depth,
                    "node": "For",
                }
            )
        self.visit(node.target)
        self.visit(node.iter)
        self._scan_block(node.body)
        self._scan_block(node.orelse)
        self._loop_depth -= 1

    def visit_While(self, node: ast.While) -> None:
        self._loop_depth += 1
        if self._loop_depth >= 2:
            self.nested_loops.append(
                {
                    "line": node.lineno,
                    "depth": self._loop_depth,
                    "node": "While",
                }
            )
        self.visit(node.test)
        self._scan_block(node.body)
        self._scan_block(node.orelse)
        self._loop_depth -= 1


def run_static_analysis(user_code: str) -> dict[str, Any]:
    """Run syntax, AST checks, cyclomatic complexity, and pylint."""
    report: dict[str, Any] = {
        "syntax_error": None,
        "unreachable_code": [],
        "nested_loops": [],
        "complexity": [],
        "pylint": [],
    }

    try:
        syntax_tree = ast.parse(user_code)
    except SyntaxError as exc:
        report["syntax_error"] = {
            "line": exc.lineno,
            "offset": exc.offset,
            "message": exc.msg,
        }
        return report

    analyzer = CodeStructureAnalyzer()
    analyzer.visit(syntax_tree)
    report["unreachable_code"] = analyzer.unreachable_code
    report["nested_loops"] = analyzer.nested_loops

    try:
        from radon.complexity import cc_rank, cc_visit

        complexity_items = cc_visit(user_code)
        report["complexity"] = [
            {
                "name": item.name,
                "line": item.lineno,
                "complexity": item.complexity,
                "rank": cc_rank(item.complexity),
            }
            for item in complexity_items
            if item.complexity >= COMPLEXITY_THRESHOLD
        ]
    except Exception as exc:
        report["complexity_error"] = f"radon analysis unavailable: {exc}"

    pylint_report = run_pylint_report(user_code)
    report["pylint"] = pylint_report.get("messages", [])
    if pylint_report.get("error"):
        report["pylint_error"] = pylint_report["error"]

    return report


def execute_code_sandbox(user_code: str, timeout_seconds: int = REVIEW_TIMEOUT_SECONDS) -> dict[str, Any]:
    """Run code in a temp-file subprocess and capture stdout/stderr/traceback."""
    temp_path: str | None = None
    try:
        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False, encoding="utf-8") as temp_file:
            temp_file.write(user_code)
            temp_path = temp_file.name

        run_kwargs: dict[str, Any] = {
            "capture_output": True,
            "text": True,
            "timeout": timeout_seconds,
        }
        if os.name != "nt" and resource is not None:
            run_kwargs["preexec_fn"] = _sandbox_preexec_fn

        completed = subprocess.run([sys.executable, temp_path], **run_kwargs)
        stderr = completed.stderr or ""
        return {
            "stdout": completed.stdout or "",
            "stderr": stderr,
            "traceback": extract_traceback(stderr),
            "returncode": completed.returncode,
            "timed_out": False,
        }
    except subprocess.TimeoutExpired as exc:
        stdout = _to_text(exc.stdout)
        stderr = _to_text(exc.stderr)
        timeout_message = f"Execution timed out after {timeout_seconds} seconds."
        stderr_with_timeout = f"{stderr}\n{timeout_message}".strip()
        return {
            "stdout": stdout,
            "stderr": stderr_with_timeout,
            "traceback": extract_traceback(stderr),
            "returncode": None,
            "timed_out": True,
        }
    except Exception as exc:
        message = f"Sandbox execution failed: {exc}"
        return {
            "stdout": "",
            "stderr": message,
            "traceback": "",
            "returncode": None,
            "timed_out": False,
        }
    finally:
        if temp_path:
            Path(temp_path).unlink(missing_ok=True)


def build_code_review_prompt(
    user_task: str,
    user_code: str,
    static_report: dict[str, Any],
    execution_report: dict[str, Any],
) -> str:
    """Build the LLM prompt for structured code review feedback."""
    return f"""
You are a Python code reviewer.

Return JSON only with this exact schema:
{{
  "summary": "short overall assessment",
  "issues": [
    {{
      "category": "syntax|logic|runtime|complexity|style",
      "severity": "high|medium|low",
      "line": null,
      "problem": "what is wrong",
      "suggestion": "how to fix it"
    }}
  ],
  "improved_code": "full corrected code if needed, otherwise empty string",
  "next_steps": ["action 1", "action 2"]
}}

USER TASK:
{user_task}

USER CODE:
{user_code}

STATIC ANALYSIS:
{json.dumps(static_report, indent=2, ensure_ascii=False)}

EXECUTION OUTPUT (stdout):
{execution_report.get("stdout", "")}

EXECUTION ERRORS (stderr):
{execution_report.get("stderr", "")}

TRACEBACK (if exists):
{execution_report.get("traceback", "")}
""".strip()


def parse_code_review_response(raw_output: str) -> dict[str, Any]:
    """Parse structured JSON review from LLM output."""
    cleaned = raw_output.strip()
    cleaned = re.sub(r"^```(?:json)?\\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\\s*```$", "", cleaned)

    try:
        payload = json.loads(cleaned)
        if isinstance(payload, dict):
            return payload
    except json.JSONDecodeError:
        pass

    return {
        "summary": cleaned,
        "issues": [],
        "improved_code": "",
        "next_steps": [],
    }


@tool("rag_pipeline_tool")
def rag_pipeline_tool(question: str) -> dict[str, Any]:
    """Retrieve, rerank, and compress context in one tool call."""
    return retrieve_payload({"question": question})


@tool("code_reviewer_tool")
def code_reviewer_tool(user_task: str, user_code: str) -> dict[str, Any]:
    """Run static analysis, sandbox execution, and structured LLM code review."""
    if not user_code.strip():
        return {
            "status": "stop",
            "message": "Please provide Python code to review.",
        }

    static_report = run_static_analysis(user_code)
    execution_report = execute_code_sandbox(user_code)
    review_prompt = build_code_review_prompt(user_task, user_code, static_report, execution_report)

    llm_response = chat_llm.invoke(review_prompt)
    structured_feedback = parse_code_review_response(str(llm_response.content))

    return {
        "status": "ok",
        "user_task": user_task,
        "static_analysis": static_report,
        "execution": execution_report,
        "structured_feedback": structured_feedback,
    }


class OrchestratorState(TypedDict, total=False):
    query: str
    context: str
    tool_results: list[dict[str, Any]]
    final_answer: str
    status: str
    message: str
    tool_choice: str
    user_task: str
    user_code: str
    code_review: dict[str, Any]
    selected_course: str
    selected_parts: list[str]
    ranked_docs: list[Document]
    prompt: str


def build_initial_state(query: str) -> OrchestratorState:
    """Shared state shape used by the orchestrator graph."""
    return {
        "query": query,
        "context": "",
        "tool_results": [],
        "final_answer": "",
    }


def build_final_prompt(question: str, context: str) -> str:
    """Build the final grounded prompt for llama3.1."""
    return f"""
You are an AI learning assistant for data science course material.
Use only the provided context to answer.
If context is insufficient, say you do not have enough information.
If the question is unrelated to course material, politely explain that you can only help with course content.
If the question is ambiguous, ask a clarifying question first.
If formulas or algorithms are present in context, preserve them exactly in your answer.
Answer directly and naturally; do not start with meta-prefaces like "Based on the provided context".
Cite sources using [1], [2], etc.

Question:
{question}

Context:
{context}
""".strip()


def guard_node(state: OrchestratorState) -> OrchestratorState:
    """Validate whether the user query is clear enough before tool usage."""
    decision = guard_question(state.get("query", ""))
    if decision.get("status") != "ok":
        message = decision.get(
            "message",
            "Your question is broad or ambiguous. Please narrow it to a specific topic, section, or example.",
        )
        return {"status": "stop", "message": message, "final_answer": message}
    return {"status": "ok"}


def router_node(state: OrchestratorState) -> OrchestratorState:
    """Choose between the RAG pipeline tool and the code reviewer tool."""
    query = state.get("query", "")
    user_task, extracted_code = extract_task_and_code(query)
    provided_code = state.get("user_code", "").strip()

    user_code = provided_code or extracted_code
    if not state.get("user_task"):
        if provided_code:
            user_task = query.strip() or "Review this Python code."

    tool_choice = choose_tool_route(query, bool(user_code))
    return {
        "tool_choice": tool_choice,
        "user_task": user_task,
        "user_code": user_code,
    }


def rag_agent_node(state: OrchestratorState) -> OrchestratorState:
    """Call the single RAG pipeline tool (retrieval + rerank + compression)."""
    tool_output = rag_pipeline_tool.invoke({"question": state.get("query", "")})

    tool_results = list(state.get("tool_results", []))
    tool_results.append({"tool": "rag_pipeline_tool", "result": tool_output})

    if tool_output.get("status") != "ok":
        message = tool_output.get("message", "I do not have enough retrieved context to answer this reliably.")
        return {
            "tool_results": tool_results,
            "status": "stop",
            "message": message,
            "final_answer": message,
        }

    return {
        "tool_results": tool_results,
        "status": "ok",
        "context": tool_output["context"],
        "selected_course": tool_output["selected_course"],
        "selected_parts": tool_output["selected_parts"],
        "ranked_docs": tool_output["ranked_docs"],
    }


def code_reviewer_node(state: OrchestratorState) -> OrchestratorState:
    """Call code reviewer tool and return structured feedback."""
    user_task = state.get("user_task", state.get("query", "")).strip() or "Review this Python code."
    user_code = state.get("user_code", "").strip()

    tool_results = list(state.get("tool_results", []))

    if not user_code:
        message = "Please provide Python code in a fenced block so I can run static analysis and sandbox execution."
        tool_results.append({"tool": "code_reviewer_tool", "result": {"status": "stop", "message": message}})
        return {
            "tool_results": tool_results,
            "status": "stop",
            "message": message,
            "final_answer": message,
        }

    tool_output = code_reviewer_tool.invoke({"user_task": user_task, "user_code": user_code})
    tool_results.append({"tool": "code_reviewer_tool", "result": tool_output})

    if tool_output.get("status") != "ok":
        message = tool_output.get("message", "Code review could not be completed.")
        return {
            "tool_results": tool_results,
            "status": "stop",
            "message": message,
            "final_answer": message,
        }

    structured_feedback = tool_output.get("structured_feedback", {})
    final_answer = json.dumps(structured_feedback, indent=2, ensure_ascii=False)

    return {
        "tool_results": tool_results,
        "status": "ok",
        "final_answer": final_answer,
        "code_review": tool_output,
    }


def answer_node(state: OrchestratorState) -> OrchestratorState:
    """Generate final response from retrieved context."""
    prompt = build_final_prompt(state.get("query", ""), state.get("context", ""))
    response = chat_llm.invoke(prompt)
    return {
        "prompt": prompt,
        "final_answer": response.content.strip(),
        "status": "ok",
    }


def route_after_guard(state: OrchestratorState) -> str:
    """Branch after guard node."""
    return "router" if state.get("status") == "ok" else "finish"


def route_after_router(state: OrchestratorState) -> str:
    """Branch after router node."""
    return "code_reviewer" if state.get("tool_choice") == "code_reviewer" else "rag_agent"


def route_after_rag(state: OrchestratorState) -> str:
    """Branch after the RAG tool node."""
    return "answer" if state.get("status") == "ok" else "finish"


orchestrator_builder = StateGraph(OrchestratorState)
orchestrator_builder.add_node("guard", guard_node)
orchestrator_builder.add_node("router", router_node)
orchestrator_builder.add_node("rag_agent", rag_agent_node)
orchestrator_builder.add_node("code_reviewer", code_reviewer_node)
orchestrator_builder.add_node("answer", answer_node)

orchestrator_builder.add_edge(START, "guard")
orchestrator_builder.add_conditional_edges(
    "guard",
    route_after_guard,
    {
        "router": "router",
        "finish": END,
    },
)
orchestrator_builder.add_conditional_edges(
    "router",
    route_after_router,
    {
        "code_reviewer": "code_reviewer",
        "rag_agent": "rag_agent",
    },
)
orchestrator_builder.add_conditional_edges(
    "rag_agent",
    route_after_rag,
    {
        "answer": "answer",
        "finish": END,
    },
)
orchestrator_builder.add_edge("answer", END)
orchestrator_builder.add_edge("code_reviewer", END)

orchestrator_graph = orchestrator_builder.compile()


def format_graph_output(state: OrchestratorState) -> dict[str, Any]:
    """Return concise structured output for inspection."""
    if state.get("tool_choice") == "code_reviewer" or state.get("code_review"):
        code_review = state.get("code_review", {})
        return {
            "route": "code_reviewer",
            "answer": state.get("final_answer", state.get("message", "")),
            "structured_feedback": code_review.get("structured_feedback", {}),
            "static_analysis": code_review.get("static_analysis", {}),
            "execution": code_review.get("execution", {}),
        }

    if state.get("status") != "ok":
        return {"answer": state.get("final_answer", state.get("message", ""))}

    ranked_docs = state.get("ranked_docs", [])
    sources = [
        {
            "course": doc.metadata.get("course", ""),
            "section": doc.metadata.get("section", ""),
            "source_file": doc.metadata.get("source_file", ""),
            "chunk_id": doc.metadata.get("chunk_id", ""),
        }
        for doc in ranked_docs
    ]

    return {
        "route": "rag_agent",
        "answer": state.get("final_answer", ""),
        "selected_course": state.get("selected_course", ""),
        "selected_parts": state.get("selected_parts", [])[:5],
        "sources": sources,
    }


class GraphRunnable:
    """Small adapter to keep existing notebook invocation style."""

    def invoke(self, input_payload: str | dict[str, str]) -> dict[str, Any]:
        if isinstance(input_payload, dict):
            query = input_payload.get("query") or input_payload.get("user_task", "")
            initial_state = build_initial_state(query)
            if input_payload.get("user_task"):
                initial_state["user_task"] = input_payload["user_task"]
            if input_payload.get("user_code"):
                initial_state["user_code"] = input_payload["user_code"]
        else:
            initial_state = build_initial_state(input_payload)

        final_state = orchestrator_graph.invoke(initial_state)
        return format_graph_output(final_state)


rag_chain = GraphRunnable()

In [17]:
def extract_markdown_code(text: str) -> str | None:
    """Extract first fenced code block, preferring python fences."""
    pattern = r"```(?:python)?\s*(.*?)```"
    match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
    if not match:
        return None

    code = match.group(1).strip()
    return code or None


def _earliest_python_keyword_start(text: str) -> int:
    """Find earliest index where Python-looking code begins."""
    python_keywords = ["def ", "class ", "import ", "for ", "while ", "if ", "print("]
    lowered = text.lower()

    positions = [lowered.find(keyword) for keyword in python_keywords if lowered.find(keyword) != -1]
    return min(positions) if positions else -1


def looks_like_python(text: str) -> bool:
    """Heuristic check for Python-like content."""
    python_keywords = ["def ", "class ", "import ", "for ", "while ", "if ", "print("]
    lowered = text.lower()
    return any(keyword in lowered for keyword in python_keywords)


def extract_task_and_code(query: str) -> tuple[str, str]:
    """Extract user task and code from markdown or plain text prompts."""
    markdown_code = extract_markdown_code(query)
    if markdown_code:
        task = re.sub(r"```(?:python)?\s*.*?```", "", query, flags=re.DOTALL | re.IGNORECASE).strip()
        return (task or "Review this Python code.", markdown_code)

    start_idx = _earliest_python_keyword_start(query)
    if start_idx != -1:
        code_candidate = query[start_idx:].strip()
        if looks_like_python(code_candidate):
            task = query[:start_idx].strip()
            return (task or "Review this Python code.", code_candidate)

    return (query.strip() or "Review this Python code.", "")

In [18]:
class RagState(TypedDict, total=False):
    query: str
    context: str
    final_answer: str
    status: str
    message: str
    selected_course: str
    selected_parts: list[str]
    ranked_docs: list[Document]


class CodeReviewState(TypedDict, total=False):
    query: str
    user_task: str
    user_code: str
    final_answer: str
    status: str
    message: str
    structured_feedback: dict[str, Any]
    static_analysis: dict[str, Any]
    execution: dict[str, Any]


def build_rag_initial_state(query: str) -> RagState:
    return {
        "query": query,
        "context": "",
        "final_answer": "",
        "status": "ok",
    }


def rag_guard_node(state: RagState) -> RagState:
    decision = guard_question(state.get("query", ""))
    if decision.get("status") != "ok":
        message = decision.get(
            "message",
            "Your question is broad or ambiguous. Please narrow it to a specific topic, section, or example.",
        )
        return {
            "status": "stop",
            "message": message,
            "final_answer": message,
        }
    return {"status": "ok"}


def rag_retrieve_node(state: RagState) -> RagState:
    tool_output = rag_pipeline_tool.invoke({"question": state.get("query", "")})
    if tool_output.get("status") != "ok":
        message = tool_output.get("message", "I do not have enough retrieved context to answer this reliably.")
        return {
            "status": "stop",
            "message": message,
            "final_answer": message,
        }

    return {
        "status": "ok",
        "context": tool_output["context"],
        "selected_course": tool_output["selected_course"],
        "selected_parts": tool_output["selected_parts"],
        "ranked_docs": tool_output["ranked_docs"],
    }


def rag_answer_node(state: RagState) -> RagState:
    prompt = build_final_prompt(state.get("query", ""), state.get("context", ""))
    response = chat_llm.invoke(prompt)
    return {
        "status": "ok",
        "final_answer": response.content.strip(),
    }


def route_after_rag_guard(state: RagState) -> str:
    return "rag_retrieve" if state.get("status") == "ok" else "finish"


def route_after_rag_retrieve(state: RagState) -> str:
    return "rag_answer" if state.get("status") == "ok" else "finish"


rag_chain_builder = StateGraph(RagState)
rag_chain_builder.add_node("rag_guard", rag_guard_node)
rag_chain_builder.add_node("rag_retrieve", rag_retrieve_node)
rag_chain_builder.add_node("rag_answer", rag_answer_node)

rag_chain_builder.add_edge(START, "rag_guard")
rag_chain_builder.add_conditional_edges(
    "rag_guard",
    route_after_rag_guard,
    {
        "rag_retrieve": "rag_retrieve",
        "finish": END,
    },
)
rag_chain_builder.add_conditional_edges(
    "rag_retrieve",
    route_after_rag_retrieve,
    {
        "rag_answer": "rag_answer",
        "finish": END,
    },
)
rag_chain_builder.add_edge("rag_answer", END)

rag_chain = rag_chain_builder.compile()


def format_rag_output(state: RagState) -> dict[str, Any]:
    if state.get("status") != "ok":
        return {
            "route": "rag_agent",
            "answer": state.get("final_answer", state.get("message", "")),
        }

    ranked_docs = state.get("ranked_docs", [])
    sources = [
        {
            "course": doc.metadata.get("course", ""),
            "section": doc.metadata.get("section", ""),
            "source_file": doc.metadata.get("source_file", ""),
            "chunk_id": doc.metadata.get("chunk_id", ""),
        }
        for doc in ranked_docs
    ]

    return {
        "route": "rag_agent",
        "answer": state.get("final_answer", ""),
        "selected_course": state.get("selected_course", ""),
        "selected_parts": state.get("selected_parts", [])[:5],
        "sources": sources,
    }


def run_rag_query(query: str) -> dict[str, Any]:
    final_state = cast(RagState, rag_chain.invoke(build_rag_initial_state(query)))
    return format_rag_output(final_state)


def build_code_review_initial_state(
    query: str = "",
    user_task: str = "",
    user_code: str = "",
) -> CodeReviewState:
    return {
        "query": query,
        "user_task": user_task,
        "user_code": user_code,
        "final_answer": "",
        "status": "ok",
    }


def code_review_prepare_node(state: CodeReviewState) -> CodeReviewState:
    query = state.get("query", "")
    existing_task = state.get("user_task", "").strip()
    existing_code = state.get("user_code", "").strip()

    extracted_task, extracted_code = extract_task_and_code(query)
    user_task = existing_task or extracted_task or "Review this Python code."
    user_code = existing_code or extracted_code

    if not user_code:
        message = "Please provide Python code in a fenced block so I can run static analysis and sandbox execution."
        return {
            "status": "stop",
            "message": message,
            "final_answer": message,
        }

    return {
        "status": "ok",
        "user_task": user_task,
        "user_code": user_code,
    }


def code_review_run_node(state: CodeReviewState) -> CodeReviewState:
    tool_output = code_reviewer_tool.invoke(
        {
            "user_task": state.get("user_task", "Review this Python code."),
            "user_code": state.get("user_code", ""),
        }
    )

    if tool_output.get("status") != "ok":
        message = tool_output.get("message", "Code review could not be completed.")
        return {
            "status": "stop",
            "message": message,
            "final_answer": message,
        }

    structured_feedback = tool_output.get("structured_feedback", {})
    return {
        "status": "ok",
        "structured_feedback": structured_feedback,
        "static_analysis": tool_output.get("static_analysis", {}),
        "execution": tool_output.get("execution", {}),
        "final_answer": json.dumps(structured_feedback, indent=2, ensure_ascii=False),
    }


def route_after_code_review_prepare(state: CodeReviewState) -> str:
    return "code_review_run" if state.get("status") == "ok" else "finish"


code_reviewer_chain_builder = StateGraph(CodeReviewState)
code_reviewer_chain_builder.add_node("code_review_prepare", code_review_prepare_node)
code_reviewer_chain_builder.add_node("code_review_run", code_review_run_node)

code_reviewer_chain_builder.add_edge(START, "code_review_prepare")
code_reviewer_chain_builder.add_conditional_edges(
    "code_review_prepare",
    route_after_code_review_prepare,
    {
        "code_review_run": "code_review_run",
        "finish": END,
    },
)
code_reviewer_chain_builder.add_edge("code_review_run", END)

code_reviewer_chain = code_reviewer_chain_builder.compile()


def format_code_review_output(state: CodeReviewState) -> dict[str, Any]:
    return {
        "route": "code_reviewer",
        "answer": state.get("final_answer", state.get("message", "")),
        "structured_feedback": state.get("structured_feedback", {}),
        "static_analysis": state.get("static_analysis", {}),
        "execution": state.get("execution", {}),
    }


def run_code_review_query(user_task: str, user_code: str) -> dict[str, Any]:
    final_state = cast(
        CodeReviewState,
        code_reviewer_chain.invoke(
            build_code_review_initial_state(user_task=user_task, user_code=user_code)
        ),
    )
    return format_code_review_output(final_state)


def run_code_review_from_query(query: str) -> dict[str, Any]:
    final_state = cast(
        CodeReviewState,
        code_reviewer_chain.invoke(build_code_review_initial_state(query=query)),
    )
    return format_code_review_output(final_state)

In [25]:
def build_code_review_prompt(
    user_task: str,
    user_code: str,
    static_report: dict[str, Any],
    execution_report: dict[str, Any],
) -> str:
    """Build a plain-text code-review prompt that always treats input as Python."""
    return f"""
You are a Python code reviewer.

Mandatory rules:
1. Always treat USER CODE as Python code, even if it looks like machine code, pseudocode, or another language.
2. Always provide an appropriate Python fix.
3. Respond with plain text only.
4. Do not return JSON.

Your response must include:
- A short diagnosis of what is wrong.
- A corrected Python version of the code.
- A short explanation of why the fix works.

USER TASK:
{user_task}

USER CODE:
{user_code}

STATIC ANALYSIS:
{json.dumps(static_report, indent=2, ensure_ascii=False)}

EXECUTION OUTPUT (stdout):
{execution_report.get("stdout", "")}

EXECUTION ERRORS (stderr):
{execution_report.get("stderr", "")}

TRACEBACK (if exists):
{execution_report.get("traceback", "")}
""".strip()


def parse_code_review_response(raw_output: str) -> str:
    """Normalize model response as plain text."""
    cleaned = raw_output.strip()
    cleaned = re.sub(r"^```(?:json|python|text)?\\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\\s*```$", "", cleaned)
    return cleaned.strip()


@tool("code_reviewer_tool")
def code_reviewer_tool(user_task: str, user_code: str) -> dict[str, Any]:
    """Run static analysis, sandbox execution, and plain-text LLM code review."""
    if not user_code.strip():
        return {
            "status": "stop",
            "message": "Please provide code to review.",
        }

    static_report = run_static_analysis(user_code)
    execution_report = execute_code_sandbox(user_code)
    review_prompt = build_code_review_prompt(user_task, user_code, static_report, execution_report)

    llm_response = chat_llm.invoke(review_prompt)
    review_response = parse_code_review_response(str(llm_response.content))

    return {
        "status": "ok",
        "user_task": user_task,
        "static_analysis": static_report,
        "execution": execution_report,
        "review_response": review_response,
    }


def code_review_prepare_node(state: CodeReviewState) -> CodeReviewState:
    query = state.get("query", "")
    existing_task = state.get("user_task", "").strip()
    existing_code = state.get("user_code", "").strip()

    extracted_task, extracted_code = extract_task_and_code(query)
    user_task = existing_task or extracted_task or "Review and fix this as Python code."
    user_code = existing_code or extracted_code

    # Fallback: treat any remaining input as Python code text.
    if not user_code:
        fallback_code = query.strip()
        if fallback_code:
            user_code = fallback_code

    if not user_code:
        message = "Please provide code text so I can review and fix it as Python."
        return {
            "status": "stop",
            "message": message,
            "final_answer": message,
        }

    return {
        "status": "ok",
        "user_task": user_task,
        "user_code": user_code,
    }


def code_review_run_node(state: CodeReviewState) -> CodeReviewState:
    tool_output = code_reviewer_tool.invoke(
        {
            "user_task": state.get("user_task", "Review and fix this as Python code."),
            "user_code": state.get("user_code", ""),
        }
    )

    if tool_output.get("status") != "ok":
        message = tool_output.get("message", "Code review could not be completed.")
        return {
            "status": "stop",
            "message": message,
            "final_answer": message,
        }

    review_response = str(tool_output.get("review_response", "")).strip()
    return {
        "status": "ok",
        "static_analysis": tool_output.get("static_analysis", {}),
        "execution": tool_output.get("execution", {}),
        "final_answer": review_response,
    }


code_reviewer_chain_builder = StateGraph(CodeReviewState)
code_reviewer_chain_builder.add_node("code_review_prepare", code_review_prepare_node)
code_reviewer_chain_builder.add_node("code_review_run", code_review_run_node)

code_reviewer_chain_builder.add_edge(START, "code_review_prepare")
code_reviewer_chain_builder.add_conditional_edges(
    "code_review_prepare",
    route_after_code_review_prepare,
    {
        "code_review_run": "code_review_run",
        "finish": END,
    },
)
code_reviewer_chain_builder.add_edge("code_review_run", END)

code_reviewer_chain = code_reviewer_chain_builder.compile()


def format_code_review_output(state: CodeReviewState) -> dict[str, Any]:
    return {
        "route": "code_reviewer",
        "answer": state.get("final_answer", state.get("message", "")),
        "static_analysis": state.get("static_analysis", {}),
        "execution": state.get("execution", {}),
    }


def run_code_review_query(user_task: str, user_code: str) -> dict[str, Any]:
    final_state = cast(
        CodeReviewState,
        code_reviewer_chain.invoke(
            build_code_review_initial_state(user_task=user_task, user_code=user_code)
        ),
    )
    return format_code_review_output(final_state)


def run_code_review_from_query(query: str) -> dict[str, Any]:
    final_state = cast(
        CodeReviewState,
        code_reviewer_chain.invoke(build_code_review_initial_state(query=query)),
    )
    return format_code_review_output(final_state)

In [15]:
question = "what is Gradient Descent?"
result = run_rag_query(question)

print(json.dumps(result, indent=2, ensure_ascii=False))

[chain/start] [chain:LangGraph] Entering Chain run with input:
{
  "query": "what is Gradient Descent?",
  "context": "",
  "final_answer": "",
  "status": "ok"
}
[chain/start] [chain:LangGraph > chain:rag_guard] Entering Chain run with input:
{
  "query": "what is Gradient Descent?",
  "context": "",
  "final_answer": "",
  "status": "ok"
}
[llm/start] [chain:LangGraph > chain:rag_guard > llm:ChatOllama] Entering LLM run with input:
{
  "prompts": [
    "Human: You classify whether a user question is clear enough for retrieval in a data-science learning assistant.\n\nReturn JSON only with one of these exact formats:\n{\"status\":\"ok\"}\n{\"status\":\"needs_clarification\",\"message\":\"<one short clarifying request>\"}\n\nDecision rule:\n- Choose \"ok\" if the question has a concrete topic and can reasonably be answered, even when short.\n- Choose \"needs_clarification\" only when user intent is genuinely unclear or too broad to answer usefully.\n\nQuestion:\nwhat is Gradient Descent

In [ ]:
prompt_1 = "I am learning pandas. Explain how to filter rows with multiple conditions using boolean indexing."
response_1 = run_rag_query(prompt_1)
print("Turn 1 prompt:", prompt_1)
print("\nTurn 1 answer:\n", response_1.get("answer", ""))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "I am learning pandas. Explain how to filter rows with multiple conditions using boolean indexing."
}
[chain/start] [chain:RunnableSequence > chain:guard_question] Entering Chain run with input:
{
  "input": "I am learning pandas. Explain how to filter rows with multiple conditions using boolean indexing."
}
[llm/start] [chain:RunnableSequence > chain:guard_question > llm:ChatOllama] Entering LLM run with input:
{
  "prompts": [
    "Human: You classify whether a user question is clear enough for retrieval in a data-science learning assistant.\n\nReturn JSON only with one of these exact formats:\n{\"status\":\"ok\"}\n{\"status\":\"needs_clarification\",\"message\":\"<one short clarifying request>\"}\n\nDecision rule:\n- Choose \"ok\" if the question has a concrete topic and can reasonably be answered, even when short.\n- Choose \"needs_clarification\" only when user intent is genuinely unclear or too bro

In [ ]:
prompt_2 = "Great. Build on your previous answer and show how to combine conditions with &, |, and ~ with correct parentheses."
response_2 = run_rag_query(prompt_2)
print("Turn 2 prompt:", prompt_2)
print("\nTurn 2 answer:\n", response_2.get("answer", ""))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Great. Build on your previous answer and show how to combine conditions with &, |, and ~ with correct parentheses."
}
[chain/start] [chain:RunnableSequence > chain:guard_question] Entering Chain run with input:
{
  "input": "Great. Build on your previous answer and show how to combine conditions with &, |, and ~ with correct parentheses."
}
[chain/end] [chain:RunnableSequence > chain:guard_question] s] Exiting Chain run with output:
{
  "status": "ok",
  "question": "Great. Build on your previous answer and show how to combine conditions with &, |, and ~ with correct parentheses."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch] Entering Chain run with input:
{
  "status": "ok",
  "question": "Great. Build on your previous answer and show how to combine conditions with &, |, and ~ with correct parentheses."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch > chain:RunnableLam

In [ ]:
prompt_3 = "Now compare boolean indexing versus DataFrame.query() and tell me when each one is better."
response_3 = run_rag_query(prompt_3)
print("Turn 3 prompt:", prompt_3)
print("\nTurn 3 answer:\n", response_3.get("answer", ""))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Now compare boolean indexing versus DataFrame.query() and tell me when each one is better."
}
[chain/start] [chain:RunnableSequence > chain:guard_question] Entering Chain run with input:
{
  "input": "Now compare boolean indexing versus DataFrame.query() and tell me when each one is better."
}
[chain/end] [chain:RunnableSequence > chain:guard_question] s] Exiting Chain run with output:
{
  "status": "ok",
  "question": "Now compare boolean indexing versus DataFrame.query() and tell me when each one is better."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch] Entering Chain run with input:
{
  "status": "ok",
  "question": "Now compare boolean indexing versus DataFrame.query() and tell me when each one is better."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch > chain:RunnableLambda] Entering Chain run with input:
{
  "status": "ok",
  "question": "Now compare boolean index

In [ ]:
prompt_4 = "Use the same context and give me a mini debugging checklist for when a pandas filter returns empty results."
response_4 = run_rag_query(prompt_4)
print("Turn 4 prompt:", prompt_4)
print("\nTurn 4 answer:\n", response_4.get("answer", ""))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Use the same context and give me a mini debugging checklist for when a pandas filter returns empty results."
}
[chain/start] [chain:RunnableSequence > chain:guard_question] Entering Chain run with input:
{
  "input": "Use the same context and give me a mini debugging checklist for when a pandas filter returns empty results."
}
[chain/end] [chain:RunnableSequence > chain:guard_question] s] Exiting Chain run with output:
{
  "status": "ok",
  "question": "Use the same context and give me a mini debugging checklist for when a pandas filter returns empty results."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch] Entering Chain run with input:
{
  "status": "ok",
  "question": "Use the same context and give me a mini debugging checklist for when a pandas filter returns empty results."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch > chain:RunnableLambda] Entering Chain run with

In [ ]:
prompt_5 = "Take that checklist and convert it into a short step-by-step algorithm I can follow every time."
response_5 = run_rag_query(prompt_5)
print("Turn 5 prompt:", prompt_5)
print("\nTurn 5 answer:\n", response_5.get("answer", ""))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Take that checklist and convert it into a short step-by-step algorithm I can follow every time."
}
[chain/start] [chain:RunnableSequence > chain:guard_question] Entering Chain run with input:
{
  "input": "Take that checklist and convert it into a short step-by-step algorithm I can follow every time."
}
[chain/end] [chain:RunnableSequence > chain:guard_question] s] Exiting Chain run with output:
{
  "status": "ok",
  "question": "Take that checklist and convert it into a short step-by-step algorithm I can follow every time."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch] Entering Chain run with input:
{
  "status": "ok",
  "question": "Take that checklist and convert it into a short step-by-step algorithm I can follow every time."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch > chain:RunnableLambda] Entering Chain run with input:
{
  "status": "ok",
  "question": "Take 

In [ ]:
prompt_6 = "Summarize our whole conversation in exactly 6 bullets and include one compact reusable code template."
response_6 = run_rag_query(prompt_6)
print("Turn 6 prompt:", prompt_6)
print("\nTurn 6 answer:\n", response_6.get("answer", ""))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Summarize our whole conversation in exactly 6 bullets and include one compact reusable code template."
}
[chain/start] [chain:RunnableSequence > chain:guard_question] Entering Chain run with input:
{
  "input": "Summarize our whole conversation in exactly 6 bullets and include one compact reusable code template."
}
[chain/end] [chain:RunnableSequence > chain:guard_question] s] Exiting Chain run with output:
{
  "status": "ok",
  "question": "Summarize our whole conversation in exactly 6 bullets and include one compact reusable code template."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch] Entering Chain run with input:
{
  "status": "ok",
  "question": "Summarize our whole conversation in exactly 6 bullets and include one compact reusable code template."
}
[chain/start] [chain:RunnableSequence > chain:RunnableBranch > chain:RunnableLambda] Entering Chain run with input:
{
  "status": "o

In [ ]:
run_rag_query("What is ADAM optimization?")

{'answer': 'ADAM optimization is an algorithm that combines the advantages of momentum and RMSProp [3]. It uses both the first moment (mean) and second moment (variance) of gradients to adapt the learning rate for each parameter.\n\nThe formula for ADAM is:\n\nm_t = (beta1 * m_t-1) + (1 - beta1) * g_t\nv_t = (beta2 * v_t-1) + (1 - beta2) * (g_t)^2\nm_hat_t = m_t / (1 - beta1^t)\nv_hat_t = v_t / (1 - beta2^t)\nw_t = w_t-1 - [ (lr / (v_hat_t + eps)^0.5) * m_hat_t ]\n\nwhere:\n\n* m_t: first moment (moving average of the gradient)\n* v_t: second moment (moving average of the squared gradient)\n* m_hat_t / v_hat_t: bias-corrected versions of the moments\n* g_t: gradient at step t\n* beta1: decay rate for the first moment (usually 0.9)\n* beta2: decay rate for the second moment (usually 0.999)\n* t: the current time step (used for bias correction)\n* lr: learning rate\n* eps: epsilon (tiny constant)\n\nThe advantages of ADAM include fast convergence, but it requires significant memory due t

In [ ]:
code_review_query = """What is wrong with this code def chlonga_bonga(int a,String b):\n\t return a + b"""
code_review_result = run_code_review_from_query(code_review_query)

print("Route:", code_review_result.get("route"))
print(code_review_result.get("answer", ""))

In [23]:
set_debug(False)
code_review_query = """What is wrong with this code def chlonga_bonga(int a,String b):\n\t return a + b"""
code_review_result = run_code_review_from_query(code_review_query)

In [24]:
code_review_result

{'route': 'code_reviewer',
 'answer': '{\n  "summary": "Based on the provided information, I\'ll create a JSON response with the exact schema you specified.\\n\\n**JSON Response**\\n```json\\n{\\n  \\"summary\\": \\"Code has syntax errors and type hints are not used correctly\\",\\n  \\"issues\\": [\\n    {\\n      \\"category\\": \\"syntax|logic\\",\\n      \\"severity\\": \\"high\\",\\n      \\"line\\": 1,\\n      \\"problem\\": \\"Invalid function definition: missing colon after parameter list\\",\\n      \\"suggestion\\": \\"Add a colon at the end of the function definition\\"\\n    },\\n    {\\n      \\"category\\": \\"runtime|complexity\\",\\n      \\"severity\\": \\"low\\",\\n      \\"line\\": null,\\n      \\"problem\\": \\"Function name is not descriptive and does not follow PEP 8 naming conventions\\",\\n      \\"suggestion\\": \\"Rename the function to a more descriptive name, e.g., `add_strings`\\"\\n    }\\n  ],\\n  \\"improved_code\\": \\"\\",\\n  \\"next_steps\\": [\\"Fi